# Problem Set 2
## Problem 2: Loopy Belief Propagation
### 1. Sum Product Algorithm

In [ ]:
import numpy as np

# Normal distribution over x_i in {0,1}
def phi(i):
    return [0.5, 0.5]

def sumprod(M, s, t, w, its):
    """Sum-product algorithm

    Arguments:
        M {n x n matrix} -- adjacency matrix of graph G
        s {integer} -- vertex in set A
        t {integer} -- vertex in set B
        w {n x n matrix} -- weights for each edge
        its {integer} -- number of iterations

    Returns:
        The approximate partition function attained after `its` iterations by
        plugging the appropriately normalized beliefs into the Bethe free energy
    """

    n = M.shape[0]

    # n x n x 2 matrix
    # Stores messages from factor nodes (C) to vertices (i).
    # C_i_messages[C, i, k] represents the message from the factor C
    # to the vertex i, where X_i = k
    C_i_messages = np.stack((M, M), axis=-1)

    # n x n x 2 matrix
    # Stores messages from vertices (i) to factor nodes (C).
    # i_C_messages[i, C, k] represents the message from the vertex i
    # to the factor C, where X_i = k
    i_C_messages = np.stack((M, M), axis=-1)

    for it in range(its):
        new_C_i_messages = C_i_messages.copy()

        # Messages from C to i
        for C in range(n):
            for i in range(n):
                # Vertices must be adjacent
                if M[C, i] == 1:
                    # Sum over assignments to x_C
                    sum = np.zeros((2))
                    # Consider the message C to i where X_C = x_C
                    for x_C in range(2):
                        # Potential function
                        if s == C and x_C != 1 or t == C and x_C != 0:
                            product = np.zeros((2))
                        else:
                            # Calculate potential for x_i = 0
                            xi_eq_0 = 0 if s == i else w[C, i] ** (1 if x_C != 0 else 0)
                            # Calculate potential for x_i = 1
                            xi_eq_1 = 0 if t == i else w[C, i] ** (1 if x_C != 1 else 0)
                            product = np.array([xi_eq_0, xi_eq_1])

                        # Product over k in C \ i = {C}
                        # Would be a loop if I was using cliques and not just edges
                        # We calculate the probability that the cut will be on C
                        message = np.array([product[0] * i_C_messages[C, i][1] + product[1] * i_C_messages[C, i][0], product[0] * i_C_messages[C, i][0] + product[1] * i_C_messages[C, i][1]])
                        product = product * message
                        sum = sum + product

                    # Normalize
                    new_C_i_messages[C, i] = ([0,0] if np.array_equal(sum, [0,0]) else sum / np.sum(sum))

        # Messages from i to C
        for i in range(n):
            for C in range(n):
                # Vertices must be adjacent
                if M[i, C] == 1:
                    product = phi(i)
                    # Product over messages into i
                    for k in range(n):
                        # Ignore messages from C into i
                        if M[k, i] == 1 and k != C:
                            product = product * C_i_messages[k, i]

                    # Normalize
                    i_C_messages[i, C] = [0,0] if np.array_equal(product, [0,0]) else product / np.sum(product)

        C_i_messages = new_C_i_messages.copy()

    beliefs = np.empty((n, 2))
    # Calculate beliefs for all x_i
    for i in range(n):
        belief = phi(i)
        for C in range(n):
            if M[i, C] == 1:
                belief = belief * C_i_messages[C, i]
        beliefs[i] = [0,0] if np.array_equal(belief, [0,0]) else belief / np.sum(belief)

    factor_beliefs = np.empty((n, 2))
    # Calculate beliefs for all x_C
    for C in range(n):
        factor_belief = np.ones((2))
        for k in range(n):
            if M[C, k] == 1:
                for x_C in range(2):
                    # Potential function
                    if s == C and x_C != 1 or t == C and x_C != 0:
                        product = np.ones((2))
                    else:
                        # Calculate potential for x_k = 0
                        xk_eq_0 = 1 if s == k else w[C, k] ** (1 if x_C != 0 else 0)
                        # Calculate potential for x_k = 1
                        xk_eq_1 = 1 if t == k else w[C, k] ** (1 if x_C != 1 else 0)
                        product = np.array([xk_eq_0, xk_eq_1])

                    message = np.array([product[0] * i_C_messages[C, k][1] + product[1] * i_C_messages[C, k][0], product[0] * i_C_messages[C, k][0] + product[1] * i_C_messages[C, k][1]])
                    factor_belief = factor_belief * product * message
        factor_beliefs[C] = [0,0] if np.array_equal(factor_belief, [0,0]) else factor_belief / np.sum(factor_belief)

    neighbor_counts = np.sum(M, axis=0)

    bethe_free_energy = np.sum(factor_beliefs, axis=0) + np.sum((neighbor_counts * beliefs.T).T, axis=0)

    return bethe_free_energy

In [737]:
#              a, b, c, d
M = np.array([[0, 1, 1, 0],  # a
              [1, 0, 0, 1],  # b
              [1, 0, 0, 1],  # c
              [0, 1, 1, 0]], # b
              dtype = np.float64)

#              a, b, c, d
w = np.array([[0, 1, 2, 0],  # a
              [1, 0, 0, 1],  # b
              [2, 0, 0, 1],  # c
              [0, 1, 1, 0]], # d
              dtype = np.float64)

s = 0
t = 3
its = 100

print("Approximate Partition Function:")
print(sumprod(M, s, t, w, its))

Approximate Partition Function:
[6.73333333 5.26666667]


In [738]:
M = np.array([[0, 1, 0],  # a
              [1, 0, 1],  # b
              [0, 1, 0]], # c
              dtype = np.float64)

w = np.array([[0, 3, 0],  # a
              [3, 0, 5],  # b
              [0, 5, 0]], # c
              dtype = np.float64)

s = 0
t = 2
its = 100

print("Approximate Partition Function:")
print(sumprod(M, s, t, w, its))

Approximate Partition Function:
[2.96078431 4.03921569]


In [739]:
#              a, b, c
M = np.array([[0, 1, 0],  # a
              [1, 0, 1],  # b
              [0, 1, 0]], # c
              dtype = np.float64)

#              a, b, c
w = np.array([[0, .2, 0],  # a
              [.2, 0, .9],  # b
              [0, .9, 0]], # c
              dtype = np.float64)

s = 0
t = 2
its = 100

print("Approximate Partition Function:")
print(sumprod(M, s, t, w, its))

Approximate Partition Function:
[2.85297416 4.14702584]


### 2. Max Product Algorithm

In [ ]:
import numpy as np

# Normal distribution over x_i in {0,1}
def phi(i):
    return [0.5, 0.5]

def maxprod(M, s, t, w, its):
    """Max-product algorithm

    Arguments:
        M {n x n matrix} -- adjacency matrix of graph G
        s {integer} -- vertex in set A
        t {integer} -- vertex in set B
        w {n x n matrix} -- weights for each edge
        its {integer} -- number of iterations

    Returns:
        The approximate maximum cut attained after `its` iterations by
        returning the assignment that maximizes each singleton belief
    """

    n = M.shape[0]

    # n x n x 2 matrix
    # Stores messages from factor nodes (C) to vertices (i).
    # C_i_messages[C, i, k] represents the message from the factor C
    # to the vertex i, where X_i = k
    C_i_messages = np.stack((M, M), axis=-1)

    # n x n x 2 matrix
    # Stores messages from vertices (i) to factor nodes (C).
    # i_C_messages[i, C, k] represents the message from the vertex i
    # to the factor C, where X_i = k
    i_C_messages = np.stack((M, M), axis=-1)

    for it in range(its):
        new_C_i_messages = C_i_messages.copy()

        # Messages from C to i
        for C in range(n):
            for i in range(n):
                # Vertices must be adjacent
                if M[C, i] == 1:
                    # Max over assignments to x_C
                    max = np.zeros((2))
                    # Consider the message C to i where X_C = x_C
                    prev_prod = np.zeros((2))
                    for x_C in range(2):
                        # Potential function
                        if s == C and x_C != 1 or t == C and x_C != 0:
                            product = np.zeros((2))
                        else:
                            # Calculate potential for x_i = 0
                            xi_eq_0 = 0 if s == i else w[C, i] ** (1 if x_C != 0 else 0)
                            # Calculate potential for x_i = 1
                            xi_eq_1 = 0 if t == i else w[C, i] ** (1 if x_C != 1 else 0)
                            product = np.array([xi_eq_0, xi_eq_1])

                        # Product over k in C \ i = {C}
                        # Would be a loop if I was using cliques and not just edges
                        message = np.array([product[0] * i_C_messages[C, i][1] + product[1] * i_C_messages[C, i][0], product[0] * i_C_messages[C, i][0] + product[1] * i_C_messages[C, i][1]])
                        product = product * message
                        max = np.max([product, prev_prod], axis=0)

                        prev_prod = product.copy()

                    # Normalize
                    new_C_i_messages[C, i] = ([0,0] if np.array_equal(max, [0,0]) else max / np.sum(max))

        # Messages from i to C
        for i in range(n):
            for C in range(n):
                # Vertices must be adjacent
                if M[i, C] == 1:
                    product = phi(i)
                    # Product over messages into i
                    for k in range(n):
                        # Ignore messages from C into i
                        if M[k, i] == 1 and k != C:
                            product = product * C_i_messages[k, i]

                    # Normalize
                    i_C_messages[i, C] = [0,0] if np.array_equal(product, [0,0]) else product / np.sum(product)

        C_i_messages = new_C_i_messages.copy()

    beliefs = np.empty((n, 2))
    # Calculate beliefs for all x_i
    for i in range(n):
        belief = phi(i)
        for C in range(n):
            if M[i, C] == 1:
                belief = belief * C_i_messages[C, i]
        beliefs[i] = [0,0] if np.array_equal(belief, [0,0]) else belief / np.sum(belief)

    # Convert beliefs into assignments
    return [0.5 if b[0] == b[1] else (1 if b[0] < b[1] else 0) for b in beliefs]

In [740]:
#              a, b, c, d
M = np.array([[0, 1, 1, 0],  # a
              [1, 0, 0, 1],  # b
              [1, 0, 0, 1],  # c
              [0, 1, 1, 0]], # b
              dtype = np.float64)

#              a, b, c, d
w = np.array([[0, 2, 1, 0],  # a
              [2, 0, 0, 1],  # b
              [1, 0, 0, 1],  # c
              [0, 1, 1, 0]], # d
              dtype = np.float64)

s = 0
t = 3
its = 100

print("Approximate Maximum Cut:")
print(maxprod(M, s, t, w, its))

Approximate Maximum Cut:
[1, 0, 0.5, 0]


In [741]:
M = np.array([[0, 1, 0],  # a
              [1, 0, 1],  # b
              [0, 1, 0]], # c
              dtype = np.float64)

w = np.array([[0, 3, 0],  # a
              [3, 0, 5],  # b
              [0, 5, 0]], # c
              dtype = np.float64)

s = 0
t = 2
its = 100

print("Approximate Maximum Cut:")
print(maxprod(M, s, t, w, its))

Approximate Maximum Cut:
[1, 1, 0]


In [742]:
#              a, b, c
M = np.array([[0, 1, 0],  # a
              [1, 0, 1],  # b
              [0, 1, 0]], # c
              dtype = np.float64)

#              a, b, c
w = np.array([[0, .2, 0],  # a
              [.2, 0, .9],  # b
              [0, .9, 0]], # c
              dtype = np.float64)

s = 0
t = 2
its = 100

print("Approximate Maximum Cut:")
print(maxprod(M, s, t, w, its))

Approximate Maximum Cut:
[1, 1, 0]
